In [ ]:
import os
print(os.listdir('/content'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Load datasets
sentiment = pd.read_csv('/content/fear_greed_index.csv')
trades = pd.read_csv('/content/historical_data.csv')

# Basic inspection
print("Sentiment Dataset Shape:", sentiment.shape)
print("Trades Dataset Shape:", trades.shape)

print("\nSentiment Columns:")
print(sentiment.columns)

print("\nTrades Columns:")
print(trades.columns)

In [ ]:
sentiment.head()

In [ ]:
trades.head()

In [ ]:
print(trades.columns.tolist())

In [ ]:
print(sentiment.columns.tolist())

In [ ]:
print(sentiment.isnull().sum())

In [ ]:
print(trades.isnull().sum())

In [ ]:
# Sentiment date
sentiment['date'] = pd.to_datetime(sentiment['date'])

# Trade date
trades['Timestamp IST'] = pd.to_datetime(
    trades['Timestamp IST'],
    dayfirst=True
)

trades['date'] = trades['Timestamp IST'].dt.date
trades['date'] = pd.to_datetime(trades['date'])

In [ ]:
print(sentiment['date'].head())
print(trades['date'].head())

In [ ]:
sentiment_map = {
    'Extreme Fear': 0,
    'Fear': 1,
    'Neutral': 2,
    'Greed': 3,
    'Extreme Greed': 4
}

sentiment['sentiment_score'] = sentiment['classification'].map(sentiment_map)

sentiment.head()

In [ ]:
print("Sentiment Date Range:")
print(sentiment['date'].min())
print(sentiment['date'].max())

print("\nTrade Date Range:")
print(trades['date'].min())
print(trades['date'].max())

In [ ]:
common_dates = set(sentiment['date']).intersection(set(trades['date']))

print("Common dates:", len(common_dates))

In [ ]:
sentiment_map = {
    'Extreme Fear': 0,
    'Fear': 1,
    'Neutral': 2,
    'Greed': 3,
    'Extreme Greed': 4
}

sentiment['sentiment_score'] = sentiment['classification'].map(sentiment_map)

In [ ]:
merged = pd.merge(
    trades,
    sentiment[['date', 'classification', 'value', 'sentiment_score']],
    on='date',
    how='inner'
)

print("Merged Shape:", merged.shape)
merged.head()

In [ ]:
pnl_by_sentiment = merged.groupby('classification')['Closed PnL'].agg([
    'count',
    'mean',
    'sum'
]).sort_values('mean', ascending=False)

print(pnl_by_sentiment)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

merged.groupby('classification')['Closed PnL'].mean().sort_values().plot(
    kind='bar'
)

plt.title('Average Closed PnL by Market Sentiment')
plt.ylabel('Average Closed PnL')
plt.xlabel('Sentiment')
plt.xticks(rotation=45)

plt.show()

In [ ]:
win_rate.sort_values().plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Win Rate by Sentiment')
plt.ylabel('Win Rate (%)')
plt.show()

In [ ]:
plt.savefig(
    'saved_charts/win_rate_by_sentiment.png',
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
merged['Win'] = merged['Closed PnL'] > 0

win_rate = merged.groupby('classification')['Win'].mean() * 100

print(win_rate)

In [ ]:
volume_analysis = merged.groupby('classification')['Size USD'].agg(
    ['count', 'mean', 'sum']
)

print(volume_analysis)

In [ ]:
import matplotlib.pyplot as plt

merged.groupby('classification')['Size USD'].mean().sort_values().plot(
    kind='bar',
    figsize=(8,5)
)

plt.title('Average Trade Size (USD) by Sentiment')
plt.ylabel('Average Size USD')
plt.show()

In [ ]:
plt.savefig(
    'saved_charts/trade_size_by_sentiment.png',
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
fee_analysis = merged.groupby('classification')['Fee'].agg(
    ['mean','sum']
)

print(fee_analysis)

In [ ]:
corr_df = merged[[
    'Closed PnL',
    'Size USD',
    'Fee',
    'sentiment_score'
]]

corr_df.corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.heatmap(
    corr_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Matrix')
plt.show()

In [ ]:
plt.savefig(
    'saved_charts/correlation_heatmap.png',
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
top_traders = (
    merged.groupby('Account')['Closed PnL']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_traders)

In [ ]:
top_traders.plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Top 10 Traders by Total Closed PnL')
plt.ylabel('Total Profit')
plt.show()

In [ ]:
plt.savefig(
    'saved_charts/top_traders.png',
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

sns.boxplot(
    x='classification',
    y='Closed PnL',
    data=merged
)

plt.title('PnL Distribution by Sentiment')
plt.xticks(rotation=45)
plt.show()

In [ ]:
import os

os.makedirs('saved_charts', exist_ok=True)

In [ ]:
plt.savefig(
    'saved_charts/pnl_distribution.png',
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
import os

print(os.listdir('saved_charts'))

In [ ]:
from scipy.stats import f_oneway

fear = merged[merged['classification']=='Fear']['Closed PnL']
greed = merged[merged['classification']=='Greed']['Closed PnL']
extreme_fear = merged[merged['classification']=='Extreme Fear']['Closed PnL']
extreme_greed = merged[merged['classification']=='Extreme Greed']['Closed PnL']

f_stat, p_value = f_oneway(
    fear,
    greed,
    extreme_fear,
    extreme_greed
)

print("F-statistic:", f_stat)
print("P-value:", p_value)

In [ ]:
total_profit = merged.groupby('classification')['Closed PnL'].sum()

print(total_profit.sort_values(ascending=False))

In [ ]:
plt.savefig('chart_name.png', dpi=300, bbox_inches='tight')

In [ ]:
trader_stats = merged.groupby('Account').agg({
    'Closed PnL':'sum',
    'Size USD':'mean',
    'Fee':'mean'
}).reset_index()

trader_stats.head()

In [ ]:
from sklearn.cluster import KMeans

X = trader_stats[['Closed PnL','Size USD','Fee']]

kmeans = KMeans(n_clusters=3, random_state=42)
trader_stats['Cluster'] = kmeans.fit_predict(X)

In [ ]:
merged['month'] = merged['date'].dt.to_period('M')

monthly_pnl = merged.groupby('month')['Closed PnL'].sum()

monthly_pnl.plot(figsize=(12,5))
plt.title("Monthly Profitability")
plt.show()

# Data Merging

The Fear & Greed dataset and Hyperliquid trading dataset were merged using the common date field. This resulted in a unified dataset containing trading activity alongside corresponding market sentiment information.

Merged Records: 211,218

In [ ]:
sentiment_daily = sentiment[['date','classification']]

sentiment_daily['prev'] = sentiment_daily['classification'].shift(1)

transitions = sentiment_daily.groupby(
    ['prev','classification']
).size()

print(transitions)

In [ ]:
strategy = merged.groupby('classification')['Closed PnL'].mean()
print(strategy)

In [ ]:
print("Merged rows:", len(merged))
print("Unique traders:", merged['Account'].nunique())
print("Unique coins:", merged['Coin'].nunique())

# Crypto Market Sentiment & Trader Performance Analysis

## Objective
This project analyzes the relationship between Bitcoin market sentiment (Fear & Greed Index) and trader performance using Hyperliquid historical trading data. The goal is to identify patterns in profitability, win rates, and trading behavior across different market sentiment conditions.




# Data Loading

The datasets were loaded into Pandas DataFrames for analysis. The Fear & Greed dataset contains market sentiment classifications, while the Hyperliquid dataset contains detailed trader execution records.



import pandas as pd
import numpy as np
...

# Data Cleaning & Preprocessing

The datasets were cleaned and prepared for analysis. Date columns were converted into datetime format, sentiment categories were mapped to numerical scores, and both datasets were aligned using common trading dates.



sentiment['date'] = pd.to_datetime(sentiment['date'])
...

# Data Merging

The trading dataset and sentiment dataset were merged using the common date field. This enabled sentiment information to be associated with each trading record.


merged = pd.merge(...)

# Profitability Analysis

This section evaluates how trader profitability changes across different market sentiment categories. Average Closed PnL was calculated for each sentiment regime.

# Win Rate Analysis

A win rate metric was calculated to determine the percentage of profitable trades under each market sentiment condition.


# Trade Size Analysis

Average trade size (USD) was analyzed to understand how trader risk exposure changes under different market sentiment regimes.

# Correlation Analysis

A correlation matrix was generated to examine relationships between profitability, trade size, fees, and sentiment score.

# Top Traders Analysis

The top-performing traders were identified based on total realized profit (Closed PnL). This analysis highlights the concentration of profitability among traders.

# Statistical Testing (ANOVA)

An ANOVA test was performed to determine whether profitability differs significantly across sentiment categories.

Hypothesis:

- H₀: Average profitability is equal across all sentiment categories.
- H₁: At least one sentiment category has a different average profitability.

# Key Findings

1. Extreme Greed exhibited the highest average trader profitability.
2. Win rates increased during optimistic market conditions.
3. Fear periods showed the largest average trade sizes.
4. Sentiment displayed a statistically significant impact on profitability (ANOVA p < 0.001).
5. Profitability was concentrated among a small number of traders.
6. Market sentiment can be used as a supplementary signal for trading and risk management.

# Dataset Summary

- Total Records Analyzed: 211,218
- Unique Traders: 32
- Unique Coins: 246
- Common Trading Days: 479

# Conclusion

Market sentiment influences trader behavior and profitability. While sentiment alone is not a strong linear predictor of profit, significant differences exist across sentiment regimes, making it a useful contextual indicator for trading decisions and risk management.